In [191]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [192]:
df = pd.read_csv("data/airbnb_listings_stockholm_detailed.csv")

print(df.shape)
df.head()

(4955, 79)


,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,164448,https://www.airbnb.com/rooms/164448,20250929042406,2025-09-29,city scrape,Double room in central Stockholm with Wi-Fi,I am renting out a nice double room on the top...,NaN,https://a0.muscache.com/pictures/f56d8d10-a7fa...,784312,...,4.97,4.84,4.76,NaN,f,2,0,2,0,2.63
1,220851,https://www.airbnb.com/rooms/220851,20250929042406,2025-09-29,city scrape,One room in appartement,Welcome!,Many restaurangs wery close and walkingdistanc...,https://a0.muscache.com/pictures/2085606/7a706...,412283,...,4.88,4.82,4.71,NaN,f,2,1,1,0,0.39
2,238411,https://www.airbnb.com/rooms/238411,20250929042406,2025-09-29,city scrape,Cozy apartment in central Stockholm,NaN,"Restaurants, cafés, museums, art galleries, pa...",https://a0.muscache.com/pictures/2806060/7fc68...,1250232,...,4.95,4.85,4.64,NaN,f,1,1,0,0,0.65
3,242188,https://www.airbnb.com/rooms/242188,20250929042406,2025-09-29,city scrape,Single room in central Stockholm with Wi-Fi,I am renting out a nice single room on the top...,NaN,https://a0.muscache.com/pictures/2303148/55d1e...,784312,...,4.97,4.89,4.83,NaN,f,2,0,2,0,2.68
4,273906,https://www.airbnb.com/rooms/273906,20250929042406,2025-09-29,city scrape,Penthouse in central Stockholm,NaN,NaN,https://a0.muscache.com/pictures/2881091/f5404...,1432722,...,4.75,5.00,5.00,NaN,f,2,2,0,0,0.02


In [193]:
#Rena pris
df["price"] = (
    df["price"]
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)

df["price"].describe()

count      3190.000000
mean       1690.473041
std        3704.741906
min         120.000000
25%         753.000000
50%        1200.000000
75%        2000.000000
max      112218.000000
Name: price, dtype: float64

In [194]:
upper = df["price"].quantile(0.99)

df = df[df["price"] <= upper].copy()

df["price"].describe()

count    3158.000000
mean     1502.045598
std      1061.893191
min       120.000000
25%       750.000000
50%      1195.500000
75%      1980.000000
max      6750.000000
Name: price, dtype: float64

In [195]:
df = df[
    (df["accommodates"] <= 10) &
    (df["bedrooms"] <= 10) &
    (df["beds"] <= 15)
].copy()

df[["accommodates", "bedrooms", "beds"]].describe()

,accommodates,bedrooms,beds
count,3124.000000,3124.000000,3124.000000
mean,3.431498,1.579385,2.093150
std,1.903898,1.140654,1.489841
min,1.000000,0.000000,0.000000
25%,2.000000,1.000000,1.000000
50%,3.000000,1.000000,2.000000
75%,4.000000,2.000000,3.000000
max,10.000000,7.000000,12.000000


In [196]:
# 1. Bathrooms (bara 1 värde saknas → median)
df["bathrooms"] = df["bathrooms"].fillna(df["bathrooms"].median())

# 2. Skapa has_rating (innan vi fyller för att inte tappa dem som faktiskt hade ratings)
df["has_rating"] = df["review_scores_rating"].notnull().astype(int)

# 3. Fyll rating med median för att kunna köra modellberäkningar
df["review_scores_rating"] = df["review_scores_rating"].fillna(
    df["review_scores_rating"].median()
)

# Kontroll
df.isna().sum()

id                                                0
listing_url                                       0
scrape_id                                         0
last_scraped                                      0
source                                            0
                                               ... 
calculated_host_listings_count_entire_homes       0
calculated_host_listings_count_private_rooms      0
calculated_host_listings_count_shared_rooms       0
reviews_per_month                               499
has_rating                                        0
Length: 80, dtype: int64

In [197]:
def simplify_property_type(x):
    x = x.lower()

    # specialfall
    if "boat" in x or "camper" in x:
        return "unique"

    # apartment-typer
    elif any(word in x for word in [
        "rental unit",
        "apartment",
        "condo",
        "loft",
        "serviced apartment",
        "aparthotel"
    ]):
        return "apartment"

    # resten = house
    else:
        return "house"


df["property_group"] = df["property_type"].apply(simplify_property_type)

df["property_group"].value_counts()


property_group
apartment    2392
house         724
unique          8
Name: count, dtype: int64

In [198]:
df["log_price"] = np.log1p(df["price"])

Städ klar enligt förra arbetet, testa olika steg av modeller.

In [199]:
missing_desc = (
    df["description"].fillna("").str.strip() == ""
)

df["has_description"] = (~missing_desc).astype(int)

In [200]:
df["description"] = df["description"].fillna("")

df["description_length"] = df["description"].str.len()

In [201]:
df[["description", "has_description", "description_length"]].head(10)

,description,has_description,description_length
0,I am renting out a nice double room on the top...,1,188
1,Welcome!,1,8
2,,0,0
3,I am renting out a nice single room on the top...,1,255
4,,0,0
5,Very centrally located small apartment at Slus...,1,495
6,You are welcome to the big and bright room in ...,1,244
7,"there are several rooms in the house, the smal...",1,250
8,"This unique apartment is situated in Vasastan,...",1,240
9,,0,0


In [202]:

df["name_length"] = df["name"].str.len()

In [203]:
numeric_features = [
    "accommodates",
    "bathrooms",
    "bedrooms",
    "beds",
    "minimum_nights",
    "availability_365",
    "review_scores_rating",
    "latitude",
    "longitude",
    "has_rating",
    "has_description",
    "description_length",
    "name_length"
]

categorical_features = [
    "room_type",
    "neighbourhood",
    "property_group",
    "property_type"
]

text_features = [
    "name",
    "description",
    "neighborhood_overview",
]

In [204]:
df_model = df[
    numeric_features +
    categorical_features +
    text_features +
    ["log_price"]
].copy()

In [205]:
df_model.describe()

,accommodates,bathrooms,bedrooms,beds,minimum_nights,availability_365,review_scores_rating,latitude,longitude,has_rating,has_description,description_length,name_length,log_price
count,3124.000000,3124.000000,3124.000000,3124.000000,3124.000000,3124.000000,3124.000000,3124.000000,3124.000000,3124.000000,3124.000000,3124.000000,3124.000000,3124.000000
mean,3.431498,1.259123,1.579385,2.093150,6.618758,204.593150,4.802468,59.319307,18.027621,0.840269,0.965749,364.610435,32.881882,7.084684
std,1.903898,0.630501,1.140654,1.489841,22.156236,124.520429,0.314101,0.033454,0.067805,0.366415,0.181902,162.262804,10.763302,0.683085
min,1.000000,0.000000,0.000000,0.000000,1.000000,0.000000,1.000000,59.232990,17.773110,0.000000,0.000000,0.000000,1.000000,4.795791
25%,2.000000,1.000000,1.000000,1.000000,1.000000,83.000000,4.750000,59.297267,17.993188,1.000000,1.000000,254.000000,26.000000,6.621406
50%,3.000000,1.000000,1.000000,2.000000,2.000000,229.500000,4.890000,59.318685,18.045881,1.000000,1.000000,420.000000,32.000000,7.086738
75%,4.000000,1.500000,2.000000,3.000000,4.000000,328.000000,5.000000,59.338600,18.076456,1.000000,1.000000,496.000000,42.000000,7.590978
max,10.000000,9.500000,7.000000,12.000000,500.000000,365.000000,5.000000,59.418950,18.191930,1.000000,1.000000,1000.000000,71.000000,8.817446


In [206]:
for col in text_features:
    df[col] = df[col].fillna("").astype(str)

In [207]:
for col in text_features:
    empty_pct = (
        (df[col].str.strip() == "").mean() * 100
    )

    print(f"{col}: {empty_pct:.1f}% empty")

name: 0.0% empty
description: 3.4% empty
neighborhood_overview: 64.0% empty


område overview saknas i väldigt många listings, eftersom vi har så få listings, känns det vanskligt att ta med det.

In [208]:
text_features = [
    "name",
    "description",
]

Testa modell utan text

In [209]:
df.info()

<class 'pandas.DataFrame'>
Index: 3124 entries, 0 to 4954
Data columns (total 85 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id                                            3124 non-null   int64  
 1   listing_url                                   3124 non-null   str    
 2   scrape_id                                     3124 non-null   int64  
 3   last_scraped                                  3124 non-null   str    
 4   source                                        3124 non-null   str    
 5   name                                          3124 non-null   str    
 6   description                                   3124 non-null   str    
 7   neighborhood_overview                         3124 non-null   str    
 8   picture_url                                   3124 non-null   str    
 9   host_id                                       3124 non-null   int64  
 10  host

In [214]:
df["all_text"] = (
    df["name"].fillna("") + " " +
    df["description"].fillna("")
)

text_feature = "all_text"

In [215]:
df_model = df[
    numeric_features +
    categorical_features +
    [text_feature, "log_price"]
].copy()

In [216]:
df_model[numeric_features] = df_model[numeric_features].fillna(
    df_model[numeric_features].median()
)

df_model[categorical_features] = df_model[categorical_features].fillna("missing")

df_model[text_feature] = df_model[text_feature].fillna("").astype(str)

In [217]:
df_model.isna().sum().sort_values(ascending=False).head(20)

accommodates            0
has_description         0
all_text                0
property_type           0
property_group          0
neighbourhood           0
room_type               0
name_length             0
description_length      0
has_rating              0
bathrooms               0
longitude               0
latitude                0
review_scores_rating    0
availability_365        0
minimum_nights          0
beds                    0
bedrooms                0
log_price               0
dtype: int64

In [218]:
from sklearn.model_selection import train_test_split

X = df_model[
    numeric_features +
    categorical_features +
    [text_feature]
]

y = df_model["log_price"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [219]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=3000,
    min_df=5,
    ngram_range=(1, 2)
)

In [220]:
X_train_tab = preprocessor.fit_transform(
    X_train[numeric_features + categorical_features]
)

X_test_tab = preprocessor.transform(
    X_test[numeric_features + categorical_features]
)

X_train_text = tfidf.fit_transform(
    X_train[text_feature]
)

X_test_text = tfidf.transform(
    X_test[text_feature]
)

In [221]:
X_train_combined = hstack([
    X_train_tab,
    X_train_text
])

X_test_combined = hstack([
    X_test_tab,
    X_test_text
])

print(X_train_combined.shape)
print(X_test_combined.shape)

(2499, 3113)
(625, 3113)


Nu är datan förberedd och tränar för att testa Ridge eller Keras. Börjar med Ridge.

In [222]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

ridge_model = Ridge(alpha=1.0)

ridge_model.fit(X_train_combined, y_train)

y_pred_log_ridge = ridge_model.predict(X_test_combined)

print("Ridge")
print("R² log:", r2_score(y_test, y_pred_log_ridge))
print("MAE log:", mean_absolute_error(y_test, y_pred_log_ridge))
print("RMSE log:", np.sqrt(mean_squared_error(y_test, y_pred_log_ridge)))

Ridge
R² log: 0.6377390118010393
MAE log: 0.31779234271897205
RMSE log: 0.4183835296887272


In [223]:
y_test_price = np.expm1(y_test)
y_pred_price_ridge = np.expm1(y_pred_log_ridge)

print("MAE price:", mean_absolute_error(y_test_price, y_pred_price_ridge))
print("RMSE price:", np.sqrt(mean_squared_error(y_test_price, y_pred_price_ridge)))
print("R² price:", r2_score(y_test_price, y_pred_price_ridge))

MAE price: 482.17922293610724
RMSE price: 767.5735434217299
R² price: 0.48814401354818904


Helt ok resultat med Ridge. Då testar vi lite neurala nätverk istället.

In [224]:
X_train_dl = X_train_combined.toarray()
X_test_dl = X_test_combined.toarray()

In [225]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

input_dim = X_train_dl.shape[1]

tf_model = keras.Sequential([
    layers.Input(shape=(input_dim,)),

    layers.Dense(128, activation="relu"),
    layers.Dropout(0.4),

    layers.Dense(64, activation="relu"),
    layers.Dropout(0.3),

    layers.Dense(1)
])

tf_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

In [226]:
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

history = tf_model.fit(
    X_train_dl,
    y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 15.4144 - mae: 3.1322 - val_loss: 0.8193 - val_mae: 0.7356
Epoch 2/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1.7802 - mae: 1.0665 - val_loss: 0.3471 - val_mae: 0.4370
Epoch 3/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1.5233 - mae: 0.9828 - val_loss: 0.2877 - val_mae: 0.4006
Epoch 4/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1.4477 - mae: 0.9706 - val_loss: 0.4327 - val_mae: 0.5015
Epoch 5/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1.3506 - mae: 0.9306 - val_loss: 0.2719 - val_mae: 0.4011
Epoch 6/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1.2988 - mae: 0.9132 - val_loss: 0.2883 - val_mae: 0.4147
Epoch 7/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1.2955 - mae: 0.9117 - val_loss: 0.3184 - val_mae: 0.4308
Epoch 8/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1.1959 - mae: 0.8812 - val_loss: 0.3006 - val_mae: 0.4223
Epoch 9/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1.18

In [227]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

y_pred_log_tf = tf_model.predict(X_test_dl).ravel()

print("TensorFlow")
print("R² log:", r2_score(y_test, y_pred_log_tf))
print("MAE log:", mean_absolute_error(y_test, y_pred_log_tf))
print("RMSE log:", np.sqrt(mean_squared_error(y_test, y_pred_log_tf)))

20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
TensorFlow
R² log: 0.3720933193190814
MAE log: 0.4207259572087585
RMSE log: 0.5508220886566692


In [228]:
y_pred_price_tf = np.expm1(y_pred_log_tf)

print("MAE price:", mean_absolute_error(y_test_price, y_pred_price_tf))
print("RMSE price:", np.sqrt(mean_squared_error(y_test_price, y_pred_price_tf)))
print("R² price:", r2_score(y_test_price, y_pred_price_tf))

MAE price: 647.8975191894531
RMSE price: 986.6986486309659
R² price: 0.15418212408828447


In [229]:
from sklearn.inspection import permutation_importance

In [231]:
X_test_perm = X_test_combined.toarray()

In [232]:
from sklearn.inspection import permutation_importance

result = permutation_importance(
    ridge_model,
    X_test_perm,
    y_test,
    n_repeats=5,
    random_state=42,
    scoring="r2"
)

In [233]:
cat_feature_names = (
    preprocessor
    .named_transformers_["cat"]
    .get_feature_names_out(categorical_features)
)

text_feature_names = tfidf.get_feature_names_out()

all_feature_names = (
    numeric_features +
    list(cat_feature_names) +
    list(text_feature_names)
)

In [234]:
perm_df = pd.DataFrame({
    "feature": all_feature_names,
    "importance_mean": result.importances_mean,
    "importance_std": result.importances_std
}).sort_values("importance_mean", ascending=False)

In [235]:
perm_df.head(30)

,feature,importance_mean,importance_std
0,accommodates,0.148700,0.010022
2,bedrooms,0.128839,0.002588
13,room_type_Entire home/apt,0.106315,0.011627
8,longitude,0.102570,0.017440
7,latitude,0.052941,0.009178
4,minimum_nights,0.025945,0.005384
15,room_type_Private room,0.022977,0.003894
98,property_type_Private room in rental unit,0.012990,0.002680
105,property_type_Room in hotel,0.012179,0.002211
5,availability_365,0.009521,0.001445


Låt oss göra en kul karta

In [236]:
import folium

In [237]:
stockholm_map = folium.Map(
    location=[59.33, 18.06],
    zoom_start=11
)

In [238]:
sample_df = df.sample(200, random_state=42)

In [239]:
for _, row in sample_df.iterrows():

    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=5,
        popup=f'Price: {row["price"]} kr',
        color="blue",
        fill=True,
        fill_opacity=0.6
    ).add_to(stockholm_map)

In [241]:
stockholm_map

In [243]:
map_df = X_test.copy()

map_df["actual_price"] = np.expm1(y_test)
map_df["predicted_price"] = np.expm1(y_pred_log_ridge)
map_df["error"] = map_df["predicted_price"] - map_df["actual_price"]
map_df["abs_error"] = map_df["error"].abs()

In [244]:
import folium

property_colors = {
    "apartment": "blue",
    "house": "green",
    "hotel": "purple",
    "unique_stay": "orange",
    "shared_room": "red",
}

m_property = folium.Map(
    location=[59.33, 18.06],
    zoom_start=11
)

for _, row in map_df.iterrows():
    group = row["property_group"]
    color = property_colors.get(group, "gray")

    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=5,
        color=color,
        fill=True,
        fill_opacity=0.7,
        popup=(
            f"<b>Recommended price:</b> {row['predicted_price']:.0f} kr<br>"
            f"<b>Actual price:</b> {row['actual_price']:.0f} kr<br>"
            f"<b>Error:</b> {row['error']:.0f} kr<br>"
            f"<b>Property group:</b> {group}"
        )
    ).add_to(m_property)

m_property

Använd kartfunktion för att visa vad modellen gissar rätt och fel

In [247]:
import folium
import numpy as np

map_df = X_test.copy()

map_df["actual_price"] = np.expm1(y_test)
map_df["predicted_price"] = np.expm1(y_pred_log_ridge)
map_df["abs_error"] = (map_df["predicted_price"] - map_df["actual_price"]).abs()

def error_color(abs_error):
    if abs_error < 250:
        return "green"
    elif abs_error < 750:
        return "orange"
    else:
        return "red"

m_error = folium.Map(
    location=[59.33, 18.06],
    zoom_start=11
)

for _, row in map_df.iterrows():
    color = error_color(row["abs_error"])

    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=6,
        color=color,
        fill=True,
        fill_opacity=0.75,
        popup=folium.Popup(
            html=(
                f"<b>Actual price:</b> {row['actual_price']:.0f} kr<br>"
                f"<b>Predicted price:</b> {row['predicted_price']:.0f} kr<br>"
                f"<b>Absolute error:</b> {row['abs_error']:.0f} kr<br>"
                f"<b>Property group:</b> {row['property_group']}<br>"
                f"<b>Room type:</b> {row['room_type']}"
            ),
            max_width=300
        )
    ).add_to(m_error)

m_error